# 05 — ROI2 engulfment / entrapment review
Containment metrics over time **and** per-timepoint / per-event renders, so the proposed events can be seen rather than trusted.

**All PROPOSED.** Containment is per-frame geometry: a candida object's voxel overlap with a macrophage. A real engulfment also requires **persistence** across consecutive frames — not tested here. Kernel: `canmac (pixi)`; CPU only (reads label stores + raw).

## 0 — bootstrap + load

In [ ]:
# repo-root bootstrap (OOD kernel CWD is notebooks/, not the repo root)
import sys, os, pathlib
_here = pathlib.Path.cwd()
_root = next((r for r in (_here, *_here.parents) if (r/"canmac"/"__init__.py").exists()),
             pathlib.Path("/vast/scratch/users/kriel.j/monash_lsm"))
sys.path.insert(0, str(_root)); os.chdir(_root)
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.colors as mcolors
from IPython.display import display, Markdown
from canmac.stages.engulfment import load_labels, containment
from canmac.io.reader import get_view

DATASET = "ROI2"
CODE  = {"free": 1, "partial": 2, "engulfed": 3}
CMAP  = mcolors.ListedColormap(["black", "#3b82f6", "#f59e0b", "#ef4444"])  # bg/free/partial/engulfed
df = pd.read_csv("results/ROI2/engulfment.csv")
TS = sorted(df.t.unique())
print(f"{len(df)} candida-object rows over {len(TS)} timepoints: t{TS[0]}-t{TS[-1]}")

## 1 — metrics over time

In [ ]:
# ---- Engulfment metrics over time (PROPOSED) ----
g = df.groupby("t").PROPOSED_class.value_counts().unstack(fill_value=0)
for c in ("engulfed", "partial", "free"):
    if c not in g: g[c] = 0
g = g[["engulfed", "partial", "free"]]

fig, ax = plt.subplots(2, 2, figsize=(14, 8))
# (1) stacked counts per frame
ax[0,0].stackplot(g.index, g.engulfed, g.partial, g.free,
                  labels=["engulfed","partial/entrapped","free"],
                  colors=["#ef4444","#f59e0b","#3b82f6"], alpha=.85)
ax[0,0].set_xlabel("timepoint"); ax[0,0].set_ylabel("candida objects")
ax[0,0].set_title("containment class per frame"); ax[0,0].legend(fontsize=8, loc="upper left")
# (2) engulfment fraction over time (the headline metric)
tot = g.sum(axis=1).replace(0, np.nan)
ax[0,1].plot(g.index, 100*g.engulfed/tot, "o-", ms=3, color="#ef4444", label="engulfed")
ax[0,1].plot(g.index, 100*(g.engulfed+g.partial)/tot, "s-", ms=3, color="#f59e0b",
             label="engulfed+partial")
ax[0,1].set_xlabel("timepoint"); ax[0,1].set_ylabel("% of candida objects")
ax[0,1].set_title("engulfment fraction"); ax[0,1].legend(fontsize=8)
# (3) overlap-fraction distribution — where the thresholds bite
ax[1,0].hist(df.overlap_frac, bins=40, color="tab:purple", alpha=.85)
ax[1,0].axvline(0.05, color="#f59e0b", ls="--", lw=1, label="touch_frac 0.05")
ax[1,0].axvline(0.90, color="#ef4444", ls="--", lw=1, label="engulf_frac 0.90")
ax[1,0].set_yscale("log"); ax[1,0].set_xlabel("overlap fraction (candida inside macrophage)")
ax[1,0].set_ylabel("objects (log)"); ax[1,0].set_title("threshold sensitivity"); ax[1,0].legend(fontsize=8)
# (4) phagocytic burden: candida per macrophage
bur = (df[df.PROPOSED_class != "free"].groupby(["t","best_macrophage"]).size()
       .groupby("t").agg(["mean","max"]))
ax[1,1].plot(bur.index, bur["mean"], "o-", ms=3, label="mean/macrophage")
ax[1,1].plot(bur.index, bur["max"], "^-", ms=3, label="max/macrophage")
ax[1,1].set_xlabel("timepoint"); ax[1,1].set_ylabel("associated candida objects")
ax[1,1].set_title("phagocytic burden"); ax[1,1].legend(fontsize=8)
for a in ax.ravel(): a.grid(alpha=.3)
plt.tight_layout(); plt.show()

display(df.PROPOSED_class.value_counts().rename("objects").to_frame().T)
print("PROPOSED — per-frame geometry only. A real engulfment needs PERSISTENCE across")
print("consecutive frames; a single-frame overlap can be transient contact or projection.")

## 2 — render a whole timepoint

In [ ]:
_CACHE = {}
def _frame(t):
    """Load (and cache) labels + raw for one timepoint."""
    if t not in _CACHE:
        mac_l = load_labels(DATASET, "macrophage", t)
        cand_l = load_labels(DATASET, "candida", t)
        mac_r = np.asarray(get_view(DATASET, "macrophage", t, correct=True).compute(), np.float32)
        cand_r = np.asarray(get_view(DATASET, "candida", t, correct=False).compute(), np.float32)
        _CACHE[t] = (mac_l, cand_l, mac_r, cand_r)
    return _CACHE[t]

def class_volume(t, cand_l):
    """Candida labels recoloured by their PROPOSED class."""
    cls = np.zeros_like(cand_l, np.uint8)
    for r in df[df.t == t].itertuples():
        cls[cand_l == int(r.candida)] = CODE[r.PROPOSED_class]
    return cls

def render_timepoint(t):
    """Whole-frame view: raw overlay | macrophage labels | candida by PROPOSED class (XY and XZ)."""
    mac_l, cand_l, mac_r, cand_r = _frame(t)
    cls = class_volume(t, cand_l)
    d = df[df.t == t]
    n_e = int((d.PROPOSED_class=="engulfed").sum()); n_p = int((d.PROPOSED_class=="partial").sum())
    display(Markdown(f"### t{t:03d} — {len(d)} candida | **engulfed {n_e}** | partial {n_p} | "
                     f"free {len(d)-n_e-n_p} | macrophages {int(mac_l.max())}"))
    fig, ax = plt.subplots(2, 3, figsize=(18, 9))
    for row, (proj, nm) in enumerate(((0, "XY (max over Z)"), (1, "XZ (max over Y)"))):
        rgb = np.zeros((*mac_r.max(proj).shape, 3), np.float32)
        m = mac_r.max(proj); c = cand_r.max(proj)
        rgb[..., 1] = m/ (m.max() or 1); rgb[..., 0] = c/(c.max() or 1); rgb[..., 2] = rgb[..., 0]
        ax[row,0].imshow(np.clip(rgb*2.2, 0, 1), aspect="auto")
        ax[row,0].set_title(f"raw {nm}\ngreen=macrophage magenta=candida")
        ax[row,1].imshow(mac_l.max(proj), cmap="nipy_spectral", aspect="auto")
        ax[row,1].set_title("macrophage labels")
        ax[row,2].imshow(cls.max(proj), cmap=CMAP, vmin=0, vmax=3, aspect="auto")
        ax[row,2].set_title("candida by PROPOSED class\nred=engulfed amber=partial blue=free")
        for a in ax[row]: a.axis("off")
    plt.tight_layout(); plt.show()

render_timepoint(TS[30])

## 3 — render individual events (zoomed, orthogonal views)

In [ ]:
def render_event(t, candida_id, margin=25):
    """Zoom ONE candida-macrophage pair: 3 orthogonal MIPs with both masks outlined."""
    mac_l, cand_l, mac_r, cand_r = _frame(t)
    row = df[(df.t == t) & (df.candida == candida_id)]
    if row.empty:
        print(f"candida {candida_id} not at t{t:03d}"); return
    r = row.iloc[0]
    pts = np.argwhere(cand_l == candida_id)
    lo = np.maximum(pts.min(0)-margin, 0); hi = np.minimum(pts.max(0)+margin+1, cand_l.shape)
    sl = tuple(slice(int(a), int(b)) for a, b in zip(lo, hi))
    display(Markdown(f"#### t{t:03d} candida {candida_id} -> macrophage {int(r.best_macrophage)} | "
                     f"**overlap {r.overlap_frac:.2f} -> PROPOSED {r.PROPOSED_class}** | "
                     f"{int(r.candida_vox)} voxels"))
    mr, cr = mac_r[sl], cand_r[sl]
    ml = (mac_l[sl] == int(r.best_macrophage)); cl = (cand_l[sl] == candida_id)
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    for i, (proj, nm) in enumerate(((0,"XY"), (1,"XZ"), (2,"YZ"))):
        m, c = mr.max(proj), cr.max(proj)
        rgb = np.zeros((*m.shape, 3), np.float32)
        rgb[...,1] = m/(m.max() or 1); rgb[...,0] = c/(c.max() or 1); rgb[...,2] = rgb[...,0]
        ax[i].imshow(np.clip(rgb*2.2, 0, 1), aspect="auto")
        ax[i].contour(ml.max(proj), colors="lime", linewidths=1.0)      # macrophage boundary
        ax[i].contour(cl.max(proj), colors="cyan", linewidths=1.0)      # candida boundary
        ax[i].set_title(f"{nm}  (lime=macrophage, cyan=candida)"); ax[i].axis("off")
    plt.tight_layout(); plt.show()

# strongest proposed events — confirm the candida really sits INSIDE the macrophage
top = df[df.PROPOSED_class == "engulfed"].nlargest(4, "candida_vox")
for r in top.itertuples():
    render_event(int(r.t), int(r.candida))

## 4 — browse frames / all events in a frame

In [ ]:
# Browse: edit these and re-run. Whole frames, then every event in one frame.
for t in TS[::10]:
    render_timepoint(t)

# T = TS[1]
# for r in df[(df.t == T) & (df.PROPOSED_class != "free")].itertuples():
#     render_event(int(r.t), int(r.candida))

## 5 — threshold sensitivity

In [ ]:
# Threshold sensitivity: how the classes shift if engulf/touch fractions change.
# (Recomputes from the stored overlap fractions — no re-segmentation.)
rows = []
for ef in (0.7, 0.8, 0.9, 0.95):
    for tf in (0.02, 0.05, 0.1, 0.2):
        cls = np.where(df.overlap_frac >= ef, "engulfed",
              np.where(df.overlap_frac >= tf, "partial", "free"))
        v = pd.Series(cls).value_counts()
        rows.append({"engulf_frac": ef, "touch_frac": tf,
                     "engulfed": int(v.get("engulfed", 0)), "partial": int(v.get("partial", 0)),
                     "free": int(v.get("free", 0))})
display(pd.DataFrame(rows).pivot_table(index="engulf_frac", columns="touch_frac",
        values="engulfed").style.background_gradient(cmap="Reds").format("{:.0f}")
        .set_caption("engulfed count vs thresholds"))